# r.soils.rosetta

This notebook runs the workflow from the r.soils.rosetta manual page on real
data and renders the figures used in the documentation. It imports SSURGO
soil texture and bulk density from USDA Soil Data Access (SDA) for a rural
subregion of the North Carolina sample dataset (Lake Wheeler area, southwest
Wake County), then estimates van Genuchten soil hydraulic parameters with the
ROSETTA pedotransfer model.

Requirements:

- The [NC sample dataset](https://grass.osgeo.org/sampledata/north_carolina/nc_spm_08_grass7.zip)
  unpacked into `~/grassdata`
- The *r.in.ssurgo* and *r.soils.rosetta* addons (`g.extension`)
- The [rosetta-soil](https://pypi.org/project/rosetta-soil/) package
  (`pip install rosetta-soil`)
- Network access for the SDA download

## Setup

In [ ]:
import subprocess
import sys

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

# Import GRASS packages
import grass.jupyter as gj
from grass.tools import Tools

# Start GRASS session
session = gj.init("~/grassdata/nc_spm_08_grass7/user1")
tools = Tools(session=session)

## Import SSURGO soil properties

With no `ssurgo_path`, *r.in.ssurgo* queries SDA for the current
computational region. We use the rural southwest corner of the sample
dataset's `elevation` tile and aggregate the surface layer (0-25.4 cm) to the
dominant component of each map unit. Map units without data (urban land,
water) stay NULL.

In [ ]:
tools.g_region(n=222500, s=215000, w=630000, e=640000, res=10)
tools.r_in_ssurgo(
    soils="ssurgo_soils",
    sand="sand",
    silt="silt",
    clay="clay",
    bulk_density="bd",
    ksat_r="ksat_ssurgo",
    hzdept_r=0,
    hzdepb_r=25.4,
    overwrite=True,
)

In [ ]:
# A quick look at one of the imported inputs.
tools.r_colors(map="sand", color="viridis", flags="e")
sand_map = gj.Map(width=600, use_region=True)
sand_map.d_shade(shade="elevation_shade", color="sand", brighten=30)
sand_map.d_legend(raster="sand", at=(6, 40, 90, 94), flags="b", title="% sand")
sand_map.show()

## Estimate van Genuchten parameters with ROSETTA

Supplying texture plus bulk density selects ROSETTA model code 3. The `ksat`
output is written in mm/hr so it can feed *r.sim.water* directly.

In [ ]:
tools.r_soils_rosetta(
    sand="sand",
    silt="silt",
    clay="clay",
    bulk_density="bd",
    theta_r="theta_r",
    theta_s="theta_s",
    alpha="alpha",
    n="n",
    ksat="ksat_rosetta",
    version=3,
    overwrite=True,
)

stats = tools.r_univar(map="ksat_rosetta", flags="g").keyval
print(
    f"ROSETTA Ksat [mm/hr]  min={float(stats['min']):.1f}"
    f"  mean={float(stats['mean']):.1f}  max={float(stats['max']):.1f}"
)

## Figures for the manual

Saturated hydraulic conductivity and saturated water content draped over the
shaded relief. Low-Ksat alluvial soils trace the drainage network while the
sandy interfluves conduct water quickly. These renders are saved as the
manual page figures.

In [ ]:
tools.r_colors(map="ksat_rosetta", color="viridis", flags="e")
ksat_fig = gj.Map(width=600, use_region=True, filename="r_soils_rosetta.png")
ksat_fig.d_shade(shade="elevation_shade", color="ksat_rosetta", brighten=30)
ksat_fig.d_legend(raster="ksat_rosetta", at=(6, 40, 90, 94), flags="b", title="mm/hr")
ksat_fig.d_barscale(at=(50, 6.5), flags="n")
ksat_fig.show()

In [ ]:
tools.r_colors(map="theta_s", color="plasma", flags="e")
theta_fig = gj.Map(width=600, use_region=True, filename="r_soils_rosetta_theta_s.png")
theta_fig.d_shade(shade="elevation_shade", color="theta_s", brighten=30)
theta_fig.d_legend(raster="theta_s", at=(6, 40, 90, 94), flags="b", title="cm3/cm3")
theta_fig.d_barscale(at=(50, 6.5), flags="n")
theta_fig.show()